# Policy evaluation with a neural-network value function (semi-gradient TD)

**Gridworld** $5\times 5$, discount $\gamma = 0.9$, fixed deterministic policy $\pi$.

The goal is to evaluate $v_\pi$ in two ways and compare them:

**1. Exact solution (model-based).** For a fixed policy the Bellman equation in matrix form is

$$
v_\pi = r_\pi + \gamma P_\pi v_\pi
\quad\Longrightarrow\quad
v_\pi = (I - \gamma P_\pi)^{-1} r_\pi ,
$$

where $[P_\pi]_{s s'} = p(s' \mid s, \pi(s))$ and $[r_\pi]_s$ is the expected immediate reward in $s$ under $\pi$.

**2. Neural-network approximation (model-free, TD).** We approximate $v_\pi(s) \approx \hat v(s; w)$ with a single-hidden-layer network

$$
\hat v(s; w) = W_2\, h(s) + b_2, \qquad h(s) = \tanh\!\big(W_1 \phi(s) + b_1\big),
\qquad w = \{W_1, b_1, W_2, b_2\},
$$

and update the weights from sampled transitions $(s_t, r_{t+1}, s_{t+1})$ with the **semi-gradient TD(0)** rule

$$
w_{t+1} = w_t + \alpha_t \Big( r_{t+1} + \gamma \hat v(s_{t+1}; w_t) - \hat v(s_t; w_t) \Big) \nabla_w \hat v(s_t; w_t).
$$

It is a **semi**-gradient because the target $r_{t+1} + \gamma \hat v(s_{t+1}; w_t)$ also depends on $w$, but we treat it as a constant and only differentiate $\hat v(s_t; w_t)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

import torch
import torch.nn as nn

np.random.seed(42)
torch.manual_seed(42);

<img src="Images/03_example_vf2.png" width="300">

## 1. Environment

In [ ]:
N     = 25          # states 0..24, row-major on a 5x5 grid
GAMMA = 0.9

ORANGE = {6, 7, 12, 16, 18, 21}   # reward -1 on entry
BLUE   = {17}                       # reward +1 on entry

UP, RIGHT, DOWN, LEFT, STAY = 0, 1, 2, 3, 4


def transition(s, a):
    """Deterministic T(s,a) -> (s', reward)."""
    r, c = s // 5, s % 5
    moves = {UP: (-1,0), RIGHT: (0,1), DOWN: (1,0), LEFT: (0,-1), STAY: (0,0)}
    dr, dc = moves[a]
    nr, nc = r + dr, c + dc

    if not (0 <= nr < 5 and 0 <= nc < 5):   # wall hit
        return s, -1.0

    sp     = 5 * nr + nc
    reward = +1.0 if sp in BLUE else (-1.0 if sp in ORANGE else 0.0)
    return sp, reward


# Deterministic policy from the figure
#       col0   col1   col2   col3   col4
# row0 [  ->  ] [  ->  ] [  ->  ] [  v  ] [  v  ]
# row1 [  ^   ] [  ^   ] [  ->  ] [  v  ] [  v  ]
# row2 [  ^   ] [  <-  ] [  v   ] [  -> ] [  v  ]
# row3 [  ^   ] [  ->  ] [  o   ] [  <- ] [  v  ]
# row4 [  ^   ] [  ->  ] [  ^   ] [  <- ] [  <- ]

POLICY = [
    RIGHT, RIGHT, RIGHT, DOWN,  DOWN,   # row 0
    UP,    UP,    RIGHT, DOWN,  DOWN,   # row 1
    UP,    LEFT,  DOWN,  RIGHT, DOWN,   # row 2
    UP,    RIGHT, STAY,  LEFT,  DOWN,   # row 3
    UP,    RIGHT, UP,    LEFT,  LEFT,   # row 4
]

## 2. Exact value function $v_\pi$ from the transition matrix $P_\pi$

Because the policy and the environment are deterministic, every row of $P_\pi$ has a single 1 (the next state) and $r_\pi(s)$ is the reward of that transition.

In [ ]:
def build_Ppi_rpi():
    """Build the 25x25 transition matrix P_pi and reward vector r_pi under POLICY."""
    P_pi = np.zeros((N, N))
    r_pi = np.zeros(N)
    for s in range(N):
        sp, r       = transition(s, POLICY[s])
        P_pi[s, sp] = 1.0
        r_pi[s]     = r
    return P_pi, r_pi


P_pi, r_pi = build_Ppi_rpi()

plt.figure(figsize=(6, 5))
plt.imshow(P_pi, cmap="Greys")
plt.title(r"Transition matrix $P_\pi$", fontweight="bold")
plt.xlabel("next state $s'$")
plt.ylabel("state $s$")
plt.xticks(range(0, N, 2))
plt.yticks(range(0, N, 2))
plt.colorbar(fraction=0.046)
plt.show()

print("r_pi as a 5x5 grid:")
print(r_pi.reshape(5, 5))

In [ ]:
# Closed form:  v_pi = (I - gamma P_pi)^{-1} r_pi
v_true = np.linalg.solve(np.eye(N) - GAMMA * P_pi, r_pi)

print("Exact value function v_pi:")
print(np.round(v_true.reshape(5, 5), 2))

# Check 1: iterative policy evaluation  v_{k+1} = r_pi + gamma P_pi v_k
v_iter = np.zeros(N)
for k in range(500):
    v_iter = r_pi + GAMMA * P_pi @ v_iter

print(f"\nmax |v_closed_form - v_iterative|  = {np.max(np.abs(v_true - v_iter)):.2e}")

# Check 2: Bellman residual
residual = v_true - (r_pi + GAMMA * P_pi @ v_true)
print(f"Bellman residual ||v - r - gamma P v||_inf = {np.max(np.abs(residual)):.2e}")

## 3. Features $\phi(s)$

* `coord_features`: normalised coordinates $(r/4,\, c/4) \in [0,1]^2$, the same input as in the previous notebook. The network has to *generalise* across cells.
* `onehot_features`: $e_s \in \mathbb{R}^{25}$. Used in the last section as a reference (tabular special case).

In [ ]:
def coord_features(s):
    """Normalised (row, col) of state s, both in [0, 1]."""
    return torch.tensor([(s // 5) / 4.0, (s % 5) / 4.0], dtype=torch.float32)


def onehot_features(s):
    """One-hot vector e_s of length N."""
    phi = torch.zeros(N, dtype=torch.float32)
    phi[s] = 1.0
    return phi


def feature_matrix(feature_fn):
    """Stack phi(s) for all states -> tensor of shape (N, d)."""
    return torch.stack([feature_fn(s) for s in range(N)])

## 4. Single-hidden-layer network and $\nabla_w \hat v(s; w)$

We call `backward()` on the **network output $\hat v(s;w)$ itself**, not on a loss. After that call, `p.grad` holds $\partial \hat v(s;w) / \partial p$ for every parameter block $p \in \{W_1, b_1, W_2, b_2\}$, which is exactly the $\nabla_w \hat v(s_t; w_t)$ in the TD update.

In [ ]:
def make_value_net(d_in, n_hidden):
    """v_hat(s; w) = W2 tanh(W1 phi(s) + b1) + b2."""
    return nn.Sequential(
        nn.Linear(d_in, n_hidden),   # W1, b1
        nn.Tanh(),
        nn.Linear(n_hidden, 1),      # W2, b2
    )


def v_hat_and_grad(net, phi):
    """
    Return v_hat(s; w) as a float and leave grad_w v_hat(s; w) in p.grad
    for every parameter p of the network.
    """
    net.zero_grad()
    v = net(phi).squeeze()
    v.backward()
    return v.item()


def v_hat_all(net, X):
    """v_hat(s; w) for all states, as a numpy array of length N."""
    with torch.no_grad():
        return net(X).squeeze(-1).numpy()

## 5. Semi-gradient TD(0) policy evaluation

At every step:

1. take the policy action, observe $r_{t+1}$ and $s_{t+1}$;
2. compute $\hat v(s_{t+1}; w_t)$ **without gradient** (the target is treated as a constant);
3. compute $\hat v(s_t; w_t)$ and $\nabla_w \hat v(s_t; w_t)$;
4. $\delta_t = r_{t+1} + \gamma \hat v(s_{t+1}; w_t) - \hat v(s_t; w_t)$;
5. $w \leftarrow w + \alpha_t\, \delta_t\, \nabla_w \hat v(s_t; w_t)$, done by hand, no optimiser.

**Step size.** $\alpha_t = \alpha_0 / (1 + t/\tau)$ decays slowly so that the weights settle down.

**Exploring the state space.** The policy is deterministic and leads every state into the absorbing state 17 (STAY). One long trajectory would update almost only state 17, so each episode starts from a uniformly random state and lasts only a few steps.

In [ ]:
def td0_nn_evaluation(net, X, v_true, n_episodes, steps_per_episode, alpha0, tau,
                      gamma=GAMMA, policy=POLICY, transition_fn=transition):
    """
    Semi-gradient TD(0) with the explicit update

        w <- w + alpha_t * (r + gamma v_hat(s') - v_hat(s)) * grad_w v_hat(s)

    X : tensor (N, d) with phi(s) in row s.
    Returns per-episode mean squared TD error and RMSE of v_hat against v_true.
    """
    td_history, rmse_history = [], []
    t = 0

    for episode in range(n_episodes):
        s       = np.random.randint(N)
        ep_td2  = 0.0

        for _ in range(steps_per_episode):
            a         = policy[s]
            s_next, r = transition_fn(s, a)

            # v_hat(s_{t+1}; w_t): part of the target, no gradient
            with torch.no_grad():
                v_next = net(X[s_next]).item()

            # v_hat(s_t; w_t) and grad_w v_hat(s_t; w_t) (stored in p.grad)
            v_s = v_hat_and_grad(net, X[s])

            # TD error and step size
            delta   = r + gamma * v_next - v_s
            alpha_t = alpha0 / (1.0 + t / tau)

            # w_{t+1} = w_t + alpha_t * delta_t * grad_w v_hat(s_t; w_t)
            with torch.no_grad():
                for p in net.parameters():
                    p += alpha_t * delta * p.grad

            ep_td2 += delta ** 2
            s       = s_next
            t      += 1

        td_history.append(ep_td2 / steps_per_episode)
        rmse_history.append(np.sqrt(np.mean((v_hat_all(net, X) - v_true) ** 2)))

    return np.array(td_history), np.array(rmse_history)

## 6. Train the network with $(r, c)$ input

In [ ]:
N_HIDDEN      = 10
ALPHA0        = 0.1
TAU           = 1e5
N_EPISODES    = 6000
STEPS_PER_EP  = 20

torch.manual_seed(0)
np.random.seed(0)

X_coord = feature_matrix(coord_features)
net     = make_value_net(d_in=2, n_hidden=N_HIDDEN)

td_history, rmse_history = td0_nn_evaluation(
    net, X_coord, v_true,
    n_episodes        = N_EPISODES,
    steps_per_episode = STEPS_PER_EP,
    alpha0            = ALPHA0,
    tau               = TAU,
)

v_hat = v_hat_all(net, X_coord)
err   = v_hat - v_true

num_params = sum(p.numel() for p in net.parameters())
print("Exact v_pi:")
print(np.round(v_true.reshape(5, 5), 2))
print("\nNN approximation v_hat:")
print(np.round(v_hat.reshape(5, 5), 2))
print("\nError v_hat - v_pi:")
print(np.round(err.reshape(5, 5), 2))
print(f"\nHidden neurons {N_HIDDEN} | {num_params} parameters | "
      f"MAE = {np.mean(np.abs(err)):.3f} | RMSE = {np.sqrt(np.mean(err**2)):.3f} | "
      f"max |err| = {np.max(np.abs(err)):.3f} (state {np.argmax(np.abs(err))})")

## 7. Comparison with the exact value function

### Learning curves
The mean squared TD error is what the algorithm *sees*. The RMSE against $v_\pi$ is what we actually care about, and we can only compute it here because we know the exact solution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(td_history, lw=0.8)
axes[0].set_yscale("log")
axes[0].set_title(r"Mean squared TD error $\delta_t^2$ per episode")
axes[0].set_xlabel("Episode")
axes[0].grid(alpha=0.3)

axes[1].plot(rmse_history, lw=1.2, color="C1")
axes[1].set_yscale("log")
axes[1].set_title(r"RMSE $\|\hat v - v_\pi\|$ over the 25 states")
axes[1].set_xlabel("Episode")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Value grids

In [ ]:
def plot_grid(ax, values, title, cmap="viridis", vmin=None, vmax=None):
    """Heatmap of a 5x5 value grid with the numbers written in each cell."""
    Z  = values.reshape(5, 5)
    im = ax.imshow(Z, cmap=cmap, vmin=vmin, vmax=vmax)
    for i in range(5):
        for j in range(5):
            ax.text(j, i, f"{Z[i, j]:.2f}", ha="center", va="center",
                    color="white" if (Z[i, j] < (Z.max() + Z.min()) / 2) == (cmap == "viridis") else "black",
                    fontsize=9)
    ax.set_title(title, fontweight="bold")
    ax.set_xticks(range(5))
    ax.set_yticks(range(5))
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")
    plt.colorbar(im, ax=ax, fraction=0.046)


vmin, vmax = v_true.min(), v_true.max()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
plot_grid(axes[0], v_true,         r"Exact $v_\pi$",          vmin=vmin, vmax=vmax)
plot_grid(axes[1], v_hat,          r"NN approximation $\hat v$", vmin=vmin, vmax=vmax)
plot_grid(axes[2], np.abs(err),    r"$|\hat v - v_\pi|$",       cmap="Reds")
plt.tight_layout()
plt.show()

### Surfaces

In [ ]:
rows = np.arange(1, 6)
cols = np.arange(1, 6)
C, R = np.meshgrid(cols, rows)

fig = plt.figure(figsize=(11, 5))

for k, (Z, title) in enumerate([(v_true, "True Value Function"),
                                (v_hat,  "Value Function Approximation")]):
    ax = fig.add_subplot(1, 2, k + 1, projection="3d")
    ax.plot_surface(C, R, Z.reshape(5, 5), cmap="viridis", edgecolor="black", linewidth=0.4)
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")
    ax.set_title(title, fontweight="bold")
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.set_yticks([1, 2, 3, 4, 5])
    ax.set_zlim(2.0, 11.0)
    ax.view_init(elev=40, azim=-60)

plt.tight_layout()
plt.show()

### State by state

In [ ]:
states = np.arange(N)

plt.figure(figsize=(12, 4))
plt.plot(states, v_true, "o-",  label=r"exact $v_\pi(s)$")
plt.plot(states, v_hat,  "s--", label=r"NN $\hat v(s; w)$")
plt.vlines(states, v_true, v_hat, colors="grey", alpha=0.5)
plt.xticks(states)
plt.xlabel("State s")
plt.ylabel("Value")
plt.title("Exact vs. approximate value per state", fontweight="bold")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 8. Extension: one-hot input (tabular special case)

With $\phi(s) = e_s$ every state gets its own input weights, so the network can represent any value function. The same TD update now recovers $v_\pi$ essentially exactly, but it no longer generalises between states.

In [ ]:
torch.manual_seed(0)
np.random.seed(0)

X_onehot   = feature_matrix(onehot_features)
net_onehot = make_value_net(d_in=N, n_hidden=20)

_, rmse_onehot = td0_nn_evaluation(
    net_onehot, X_onehot, v_true,
    n_episodes        = 2000,
    steps_per_episode = 20,
    alpha0            = 0.05,
    tau               = 1e5,
)

v_hat_onehot = v_hat_all(net_onehot, X_onehot)
err_onehot   = v_hat_onehot - v_true

print("NN approximation with one-hot input:")
print(np.round(v_hat_onehot.reshape(5, 5), 2))
print(f"\nMAE = {np.mean(np.abs(err_onehot)):.4f} | max |err| = {np.max(np.abs(err_onehot)):.4f}")

plt.figure(figsize=(7, 4))
plt.plot(rmse_history, label=rf"input $(r, c)$, {N_HIDDEN} neurons")
plt.plot(rmse_onehot,  label=r"input one-hot $e_s$, 20 neurons")
plt.yscale("log")
plt.xlabel("Episode")
plt.ylabel(r"RMSE $\|\hat v - v_\pi\|$")
plt.title("Effect of the input representation", fontweight="bold")
plt.legend()
plt.grid(alpha=0.3)
plt.show()